In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from pathlib import Path
from plicalib import meshes
from plicalib import io as plica_io
from plicalib.segmentation import Annotation, CurvatureSegmentation
from tqdm.auto import tqdm
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
import plotly.express as px

### Useful functions

In [ ]:
def segment_folds(
        mesh_dataset : dict,
        segmentation_results : dict,
        annotation_results : dict,
        default_params_dict : dict,
        curvature_step : float = 0.01,
        quantile_step : float = 0.01,
        distance_step : float = 0.1,
):
    fig = go.FigureWidget()
    fig.update_layout(
        title="",
        width=800, height=680,
        scene=dict(
            xaxis_title='x', yaxis_title='y', zaxis_title='z',
            aspectmode='data',
            uirevision="keep"  # preserve camera/zoom
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        legend=dict(itemsizing='constant')
    )
    fig.layout.legend.y = 0.5
    palette = px.colors.qualitative.T10
    sample_id_widget = widgets.Dropdown(options=[], description='Sample')
    show_hide_clusters_widget = widgets.Checkbox(value=True, description='Show/Hide Clusters')
    show_hide_curvature_widget = widgets.Checkbox(value=False, description='Show/Hide Curvature')
    min_H_widget = widgets.FloatText(value=default_params_dict['min_H'], description='Min H', step=curvature_step)
    max_H_widget = widgets.FloatText(value=default_params_dict['max_H'], description='Max H', step=curvature_step)
    use_pc2_widget = widgets.Checkbox(value=default_params_dict['use_pc2'], description='Use PC2')
    pc2_quantile_widget = widgets.BoundedFloatText(value=default_params_dict['pc2_quantile'], description='PC2 Quantile', step=quantile_step, min=0.0, max=1.0)
    max_num_clusters_widget = widgets.BoundedIntText(value=default_params_dict['max_num_clusters'], description='Num Clusters', min=1)
    expand_distance_widget = widgets.BoundedFloatText(value=default_params_dict['expand_distance'],  description='Expand Distance', step=distance_step, min=0.0)
    expand_graph_distance_widget = widgets.BoundedIntText(value=default_params_dict['expand_graph_distance'], description='Expand Graph Distance', min=0)
    join_method_widget = widgets.Dropdown(options=['and', 'or'], value=default_params_dict['join_method'], description='Join Method')
    clusters_selection_widget = widgets.TagsInput(value=[],allow_duplicates=False)
    close_button_widget = widgets.Button(description='Close')
    row1_widget = widgets.HBox(
        [sample_id_widget, widgets.HBox([ widgets.Label("Cluster IDs", tooltip="Save cluster indices as comma-separated values, e.g. 0,2,5. You can merge clusters using '+', e.g. [0+1,2,5]."),
                                           clusters_selection_widget]), 
                                           close_button_widget],
        layout=widgets.Layout(width="100%", justify_content="space-between", align_items="center")
    )
    row2_left_widget = widgets.VBox([
        show_hide_clusters_widget,
        show_hide_curvature_widget,
        min_H_widget, max_H_widget,
        use_pc2_widget, pc2_quantile_widget,
        max_num_clusters_widget,
        expand_distance_widget,
        expand_graph_distance_widget,
        join_method_widget
    ])
    # Box is better than VBox for “just a figure”
    row2_right_widget = widgets.Box(
        [fig],
        layout=widgets.Layout(width="100%", height="100%", overflow="visible")
    )
    # -----------------------------
    grid = widgets.GridspecLayout(2, 2, width="100%", grid_gap="12px")
    grid[0, :] = row1_widget
    grid[1, 0] = row2_left_widget
    grid[1, 1] = row2_right_widget
    grid.layout.grid_template_columns = "320px 1fr"
    grid.layout.grid_template_rows = "auto 1fr"
    grid.layout.align_items = "flex-start"
    row2_left_widget.layout = widgets.Layout(
        width="100%",
        height="680px",
        overflow_y="auto",
        overflow_x="hidden",
        align_self="flex-start"
    )
    # -----------------------------
    row2_right_widget.layout = widgets.Layout(
        width="100%",
        height="680px",
        align_self="stretch"
    )
    def _plot_mesh(delete_all):
        if delete_all:
            fig.data = []
        else:
            fig.data = tuple(t for t in fig.data if t.name != 'mesh')
        sample_id = sample_id_widget.value
        vertices, triangles = mesh_dataset[sample_id]['vertices'], mesh_dataset[sample_id]['triangles']
        mesh_trace = go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1], 
            z=vertices[:, 2],
            i=triangles[:, 0],
            j=triangles[:, 1],
            k=triangles[:, 2],
            color=palette[0],
            opacity=0.5,
            name='mesh'
        )
        if show_hide_curvature_widget.value:
            mesh_trace.intensity = segmentation_results[sample_id].vertex_mean_curvature
            mesh_trace.colorscale = 'RdBu'
            mesh_trace.cmid = 0.0
            mesh_trace.cmin, mesh_trace.cmax = np.nanquantile(segmentation_results[sample_id].vertex_mean_curvature, [0.05, 0.95])
            mesh_trace.colorbar.x = 0.01
            mesh_trace.colorbar.y = 0.5
        else:
            mesh_trace.intensity = None
        fig.add_trace(mesh_trace)
    def _show_hide_clusters(b):
        for trace in fig.data:
            if 'cluster' in trace.name:
                if show_hide_clusters_widget.value:
                    trace.visible = True
                else:
                    trace.visible = False    
    def _plot_clusters():
        name = sample_id_widget.value
        segmentation = segmentation_results[name]
        expanded_clusters = segmentation.run()
        if len(fig.data) > 0:
            fig.data = tuple(t for t in fig.data if 'cluster' not in t.name)
        for i, exanded_cluster in enumerate(expanded_clusters):
            cluster_trace = go.Scatter3d(
                x=segmentation.vertices[exanded_cluster, 0],
                y=segmentation.vertices[exanded_cluster, 1],
                z=segmentation.vertices[exanded_cluster, 2],
                mode='markers',
                marker=dict(size=2, color=palette[(i % (len(palette)-1)) + 1]),
                name=f'cluster_{i}'
            )
            cluster_trace.visible = True if show_hide_clusters_widget.value else False
            fig.add_trace(cluster_trace)   
    def _on_parameter_change(change):
        name = sample_id_widget.value
        segmentation = segmentation_results[name]
        segmentation.update_parameter('min_H', min_H_widget.value)
        segmentation.update_parameter('max_H', max_H_widget.value)
        segmentation.update_parameter('use_pc2', use_pc2_widget.value)
        segmentation.update_parameter('pc2_quantile', pc2_quantile_widget.value)
        segmentation.update_parameter('max_num_clusters', max_num_clusters_widget.value)
        segmentation.update_parameter('expand_distance', expand_distance_widget.value)
        segmentation.update_parameter('expand_graph_distance', expand_graph_distance_widget.value)
        segmentation.update_parameter('join_method', join_method_widget.value)
        
        _plot_clusters()
    def _on_annotation_change(change):
        name = sample_id_widget.value
        annotation_results[name] =  clusters_selection_widget.value
    


    def _on_sample_change(change):
        name = sample_id_widget.value
        segmentation = segmentation_results[name]
        #show_hide_clusters_widget.value = True
        #show_hide_curvature_widget.value = False
        min_H_widget.value = segmentation.params['min_H']
        max_H_widget.value = segmentation.params['max_H']
        use_pc2_widget.value = segmentation.params['use_pc2']
        pc2_quantile_widget.value = segmentation.params['pc2_quantile']
        max_num_clusters_widget.value = segmentation.params['max_num_clusters']
        expand_distance_widget.value = segmentation.params['expand_distance']
        expand_graph_distance_widget.value = segmentation.params['expand_graph_distance']
        join_method_widget.value = segmentation.params['join_method']
        if name in annotation_results and annotation_results[name] is not None:
            clusters_selection_widget.value = annotation_results[name]
        else:                
            clusters_selection_widget.value = []
        _plot_mesh(True)
        _plot_clusters()
    sample_id_widget.options = list(mesh_dataset.keys())
    sample_id_widget.observe(_on_sample_change, names='value')
    sample_id_widget.value = list(mesh_dataset.keys())[0]
    for widget in [min_H_widget, max_H_widget, use_pc2_widget, pc2_quantile_widget, max_num_clusters_widget,
                   expand_distance_widget, expand_graph_distance_widget, join_method_widget]:
        widget.observe(_on_parameter_change, names='value')

    show_hide_clusters_widget.observe(_show_hide_clusters, names='value')
    show_hide_curvature_widget.observe(lambda change: _plot_mesh(False), names='value')
    clusters_selection_widget.observe(_on_annotation_change, names='value')
    display(grid)

### Load dataset

In [ ]:
root_path = Path("examples/legdisc/")
metadata_file = root_path.joinpath("legdisc.json")
mesh_db = plica_io.from_json_database(metadata_file, separate_into_vertices_and_faces=True, root_path=root_path, load_params = {'orient_by_chull': True,  'taubin_filter_iterations': 5})

### Run segmentation/annotation

In [ ]:
default_params_dict = {
    'min_H': 0.05,
    'max_H': 1.0,
    'use_pc2': False,
    'pc2_quantile': 0.0,
    'max_num_clusters': 10,
    'expand_distance': 0.0,
    'expand_graph_distance': 0,
    'join_method': 'or',
}
segmentation_results = {}
annotation_results = {}
pbar = tqdm(mesh_db.items())
for name, fields in pbar:
    segmentation_results[name] = CurvatureSegmentation(default_params_dict,
                                                    vertices=mesh_db[name]['vertices'],
                                                    triangles=mesh_db[name]['triangles'])

In [ ]:
segment_folds(mesh_db, segmentation_results, annotation_results, default_params_dict)

In [ ]:
segmentation_results['20250207_ecadGFP_legdisc_2hAPF_disc1'].get_segmentation(annotation_results['20250207_ecadGFP_legdisc_2hAPF_disc1'])['segmentations']